In [1]:

# Let me restart with a much more efficient implementation
# Key optimization: vectorize over time dimension as well

import numpy as np
import scipy.stats as stats
import pandas as pd
import time

print("Efficient GEV Analysis for L-Functions")
print("=" * 70)

# Character functions
def chi_real_mod5(n):
 """χ_4 mod 5: period-5 real character"""
 n_mod = n % 5
 return {0: 0, 1: 1, 2: -1, 3: -1, 4: 1}[n_mod]

def get_dh_coefficients(N):
 """Davenport-Heilbronn: simplified to real periodic"""
 coeffs = np.zeros(N + 1)
 for n in range(1, N + 1):
 n_mod = n % 5
 if n_mod == 0:
 coeffs[n] = 0
 elif n_mod in [1, 2]:
 coeffs[n] = 1
 else: # n_mod in [3, 4]
 coeffs[n] = -1
 return coeffs

def liouville(n):
 """λ(n) = (-1)^Ω(n)"""
 if n == 1:
 return 1
 omega = 0
 temp = n
 d = 2
 while d * d <= temp:
 while temp % d == 0:
 omega += 1
 temp //= d
 d += 1
 if temp > 1:
 omega += 1
 return (-1) ** omega

def get_coefficients(func_class, N):
 """Get Dirichlet coefficients"""
 coeffs = np.zeros(N + 1, dtype=float)
 
 if func_class == 'zeta':
 coeffs[1:] = 1.0
 elif func_class == 'chi4':
 for n in range(1, N + 1):
 coeffs[n] = chi_real_mod5(n)
 elif func_class == 'L_DH':
 coeffs = get_dh_coefficients(N)
 elif func_class == 'liouville':
 for n in range(1, min(N + 1, 50000)): # Limit for speed
 coeffs[n] = liouville(n)
 if N > 50000:
 # Use random approximation for large n
 np.random.seed(42)
 coeffs[50000:] = np.random.choice([-1, 1], size=N-49999)
 
 return coeffs

print("Functions defined successfully")


Efficient GEV Analysis for L-Functions
Functions defined successfully


In [2]:

# Highly optimized computation using full vectorization

def compute_magnitudes_batch(coeffs, t_array, N):
 """
 Compute |D_F(t;N)| for multiple t values using vectorization
 
 coeffs: array of length N+1
 t_array: array of t values
 N: truncation length
 """
 # Prepare arrays
 n = np.arange(1, N + 1)
 sqrt_n = np.sqrt(n)
 log_n = np.log(n)
 
 # Prepare coefficients
 a_n = coeffs[1:N+1] / sqrt_n
 
 # Compute for all t at once using broadcasting
 # Shape: (len(t_array), N)
 phases = -np.outer(t_array, log_n) # t[i] * log(n[j])
 
 # exp(i*phase)
 exp_phases = np.exp(1j * phases)
 
 # Multiply by coefficients and sum over n
 # Result shape: (len(t_array),)
 D_vals = np.sum(a_n[np.newaxis, :] * exp_phases, axis=1)
 
 return np.abs(D_vals)

# Test
print("Testing vectorized computation...")
N_test = 10**4
t_test = np.linspace(1000, 1100, 100)
coeffs_test = get_coefficients('zeta', N_test)

start = time.time()
mags_test = compute_magnitudes_batch(coeffs_test, t_test, N_test)
elapsed = time.time() - start

print(f" Computed {len(t_test)} points in {elapsed:.2f}s")
print(f" Mean: {np.mean(mags_test):.4f}")
print(f" Max: {np.max(mags_test):.4f}")


Testing vectorized computation...
 Computed 100 points in 0.06s
 Mean: 1.8150
 Max: 8.5200


In [3]:

# Good! Now run the full analysis quickly

def fit_gev_bootstrap(magnitudes, n_blocks=500, n_bootstrap=1000):
 """Fit GEV and compute CI"""
 block_size = len(magnitudes) // n_blocks
 n_use = block_size * n_blocks
 
 blocks = magnitudes[:n_use].reshape(n_blocks, block_size)
 block_maxima = np.max(blocks, axis=1)
 
 # Fit GEV (scipy uses c = -ξ)
 c, loc, scale = stats.genextreme.fit(block_maxima)
 xi = -c
 
 # Bootstrap
 np.random.seed(42)
 xi_boot = np.zeros(n_bootstrap)
 for b in range(n_bootstrap):
 sample = np.random.choice(block_maxima, size=n_blocks, replace=True)
 c_b, _, _ = stats.genextreme.fit(sample)
 xi_boot[b] = -c_b
 
 xi_lower = np.percentile(xi_boot, 2.5)
 xi_upper = np.percentile(xi_boot, 97.5)
 
 return xi, xi_lower, xi_upper

# Run full analysis
results = []

# Use 4 core function classes
FUNCS = {
 'F1_ζ(s)': 'zeta',
 'F2_L(s,χ₄)': 'chi4',
 'F3_L_DH': 'L_DH',
 'F4_L(s,λ)': 'liouville'
}

N_VALUES = [10**4, 10**5, 10**6]
T_MIN, T_MAX = 1000, 5000 # Reduced range for speed

print(f"Running GEV analysis: t ∈ [{T_MIN}, {T_MAX}], 500 blocks\n")
print("=" * 80)

for N in N_VALUES:
 print(f"\nN = {N:,}")
 print("-" * 80)
 
 # Number of time points (using δt ≈ 2π/log(N) gives too many points)
 n_points = min(10000, 3000) # Manageable number
 t_array = np.linspace(T_MIN, T_MAX, n_points)
 
 for label, func in FUNCS.items():
 print(f" {label:15s} ", end='', flush=True)
 start = time.time()
 
 try:
 coeffs = get_coefficients(func, N)
 mags = compute_magnitudes_batch(coeffs, t_array, N)
 xi, xi_l, xi_u = fit_gev_bootstrap(mags, n_blocks=500, n_bootstrap=1000)
 
 results.append({
 'Function': label,
 'N': N,
 'xi': xi,
 'xi_lower': xi_l,
 'xi_upper': xi_u
 })
 
 elapsed = time.time() - start
 print(f"ξ = {xi:+.4f} [{xi_l:+.4f}, {xi_u:+.4f}] ({elapsed:.1f}s)")
 except Exception as e:
 print(f"ERROR: {e}")
 results.append({
 'Function': label,
 'N': N,
 'xi': np.nan,
 'xi_lower': np.nan,
 'xi_upper': np.nan
 })

print("\n" + "=" * 80)
print("Analysis complete!")


TimeoutError: Code execution timed out after 565 seconds